# Source text 점검
합성 PDF와 그래프 CSV를 만들어 모듈을 실행하고 결과 요약을 확인합니다.

In [ ]:
from pathlib import Path
import csv, hashlib, tempfile
import fitz
from payment_card.file_pipeline.graph.source_text import build
from payment_card.file_pipeline.graph.export import CSV_COLUMNS

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/card_benefit').exists()), Path.cwd())
WORK = Path(tempfile.mkdtemp(prefix="source-text-check-", dir="/tmp"))


## 입력 예시 만들기

In [ ]:
src=WORK/"input"; src.mkdir(); pdf=fitz.open(); page=pdf.new_page(); page.insert_text((72,72),"안내\n월 10,000원\n"); pdf.save(WORK/"synthetic.pdf"); pdf.close()
sha=hashlib.sha256(b"synthetic").hexdigest()
with (WORK/"documents.csv").open("w",encoding="utf-8",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["sha256","file_path","canonical"]); w.writeheader(); w.writerow({"sha256":sha,"file_path":str(WORK/"synthetic.pdf"),"canonical":"Y"})
rows={"fp_nodes":[{"id":"b","node_type":"Benefit","props":"{}","chunk_id":"c","run_id":"r"}],"fp_edges":[],"fp_evidence":[{"node_id":"b","edge_id":"","doc_sha256":sha,"page_no":"1","start_pos":"1","end_pos":"4","quote_status":"located"}],"fp_load_runs":[{"run_id":"r","input_folder_name":"synthetic","model_name":"","started_at":"","node_count":"1","edge_count":"0"}]}
for table,items in rows.items():
    with (src/f"{table}.csv").open("w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=CSV_COLUMNS[table]); w.writeheader(); w.writerows(items)

## 모듈 호출 및 결과 표시

In [ ]:
report=build(src,"synthetic",WORK/"documents.csv",WORK/"output",max_chars=700,margin_lines=1)
report

In [ ]:
with (WORK/"output/fp_nodes.csv").open(encoding="utf-8") as f:
    result=list(csv.DictReader(f))
result[0]["props"]

## Trace JSON 입력 확인

In [ ]:
import json
trace_dir=WORK/"trace"; trace_issuer=trace_dir/"synthetic"; trace_issuer.mkdir(parents=True,exist_ok=True)
trace_text=fitz.open(WORK/"synthetic.pdf")[0].get_text()
(trace_issuer/"synthetic-doc.json").write_text(json.dumps({"document":{"sha256":sha},"pages":[{"page_no":1,"text":trace_text}]}),encoding="utf-8")
trace_report=build(src,"synthetic",WORK/"documents.csv",WORK/"trace-output",trace_dir=trace_dir)
trace_report["source_counts"]